# Federated Learning – Run All Experiments (Colab)

Everything runs from the **GitHub codebase**: clone (or pull), install, download data, then run all experiments. When you hit trouble, fix in Cursor → push → here run **Cell 1** again (`git pull`) and continue.

1. **Runtime → Change runtime type → GPU (T4)**
2. Set `REPO_URL` in Cell 1 to your repo.
3. Run all cells.

## 1. Clone from GitHub & install

In [ ]:
import os

REPO_URL = "https://github.com/masud1901/Federated_learning_research.git"
PROJECT_DIR = "/content/Federated_learning_research"

if os.path.isdir(PROJECT_DIR):
    get_ipython().run_line_magic('cd', PROJECT_DIR)
    get_ipython().system('git pull')
else:
    get_ipython().system(f'git clone {REPO_URL} {PROJECT_DIR}')
    get_ipython().run_line_magic('cd', PROJECT_DIR)

get_ipython().run_line_magic('pip', 'install -q timm scikit-learn matplotlib numpy pandas psutil scipy Pillow')

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("Project:", PROJECT_DIR)

## 2. Paths & datasets (Kaggle or /content)

In [ ]:
import os

# Data and results on /content (or set USE_DRIVE=True and mount Drive first)
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATASET_ROOT = "/content/drive/MyDrive/datasets"
    RESULTS_DIR  = "/content/drive/MyDrive/fl_results"
else:
    DATASET_ROOT = "/content/datasets"
    RESULTS_DIR  = "/content/results"

# Main dataset: prepared from raw; all experiments load from here
MAIN_DATASET_ROOT = os.path.join(os.path.dirname(DATASET_ROOT.rstrip('/')), 'main_dataset')
os.makedirs(DATASET_ROOT, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.environ["COLAB_RESULTS_DIR"] = RESULTS_DIR
print("Raw (downloads):", DATASET_ROOT)
print("Main dataset (for experiments):", MAIN_DATASET_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)

### 2b. Download datasets from Kaggle

Add **KAGGLE_USERNAME** and **KAGGLE_KEY** in Colab **Secrets** (key icon in left sidebar), then run this cell.

In [ ]:
import os

USE_KAGGLE = True
if USE_KAGGLE:
    try:
        from google.colab import userdata
        u = userdata.get('KAGGLE_USERNAME')
        k = userdata.get('KAGGLE_KEY')
        os.makedirs("/root/.kaggle", exist_ok=True)
        with open("/root/.kaggle/kaggle.json", "w") as f:
            f.write(f'{{"username":"{u}","key":"{k}"}}')
        os.chmod("/root/.kaggle/kaggle.json", 0o600)
    except Exception as e:
        print("Secrets?", e)
        USE_KAGGLE = False

if USE_KAGGLE:
    get_ipython().system('pip install -q kaggle')
    get_ipython().system(f'kaggle datasets download uraninjo/augmented-alzheimer-mri-dataset -p {DATASET_ROOT} --unzip')
    get_ipython().system(f'kaggle datasets download alemranp/ratinal-deasis -p {DATASET_ROOT} --unzip')
    get_ipython().system(f'kaggle datasets download tawsifurrahman/tuberculosis-tb-chest-xray-dataset -p {DATASET_ROOT} --unzip')
    for old, new in [("augmented-alzheimer-mri-dataset", "AugmentedAlzheimerDataset"), ("ratinal-deasis", "Ratinal_Deasis"), ("tuberculosis-tb-chest-xray-dataset", "TB_Chest_Radiography_Database")]:
        p_old = os.path.join(DATASET_ROOT, old)
        p_new = os.path.join(DATASET_ROOT, new)
        if os.path.isdir(p_old) and not os.path.isdir(p_new):
            os.rename(p_old, p_new)
    print("Raw datasets at", DATASET_ROOT)
else:
    print("Skipped Kaggle. Put raw data in", DATASET_ROOT)

### 2c. Build main dataset (run experiments from here)

Prepares a single canonical layout (Alzheimer, Retinal, TB) from raw downloads. All experiments load from this main dataset.

In [ ]:
get_ipython().run_line_magic('cd', PROJECT_DIR)
get_ipython().system(f'python scripts/prepare_main_dataset.py --raw-dir {DATASET_ROOT} --main-dir {MAIN_DATASET_ROOT}')
os.environ["COLAB_DATASET_ROOT"] = MAIN_DATASET_ROOT
print("Experiments will use:", os.environ["COLAB_DATASET_ROOT"])

## 3. Quick test (from codebase)

In [ ]:
get_ipython().run_line_magic('cd', PROJECT_DIR)
get_ipython().system('python main.py --rounds 3 --local-epochs 1 --max-samples 50 --method FedAvg')

## 4. Run all paper experiments (from codebase)

In [ ]:
get_ipython().run_line_magic('cd', PROJECT_DIR)
get_ipython().system('python run_all_experiments.py --seeds 3 --rounds 20 --local-epochs 3')

## When you hit trouble

1. Fix code in **Cursor**, push to GitHub.
2. Here, run **Cell 1** again (`git pull`).
3. Re-run from Cell 3 or 4. All runs use the codebase from the repo.